In [1]:
import json
import requests
from urllib.parse import urljoin
import re

class ReproducibilityChecker:
    def __init__(self, json_file):
        self.repositories = self.load_metadata(json_file)

    def load_metadata(self, json_file):
        with open(json_file, 'r') as file:
            data = json.load(file)
        return {artifact_id: artifact['uri'] for artifact_id, artifact in data['artifacts'].items()}

    def transform_to_raw_github(self, uri, filename):
        if "github.com" in uri:
            parts = uri.strip("/").split("/")
            if len(parts) >= 5:
                user, repo = parts[3], parts[4]
                return f"https://raw.githubusercontent.com/{user}/{repo}/main/{filename}"
        return urljoin(uri + '/', filename)

    def check_file_exists(self, base_uri, filename):
        raw_uri = self.transform_to_raw_github(base_uri, filename)
        try:
            response = requests.head(raw_uri, timeout=10)
            return response.status_code == 200
        except requests.RequestException:
            return False

    def check_contains_filetype(self, base_uri, pattern):
        if "github.com" not in base_uri:
            return False
        parts = base_uri.strip("/").split("/")
        if len(parts) < 5:
            return False
        user, repo = parts[3], parts[4]
        api_url = f"https://api.github.com/repos/{user}/{repo}/git/trees/main?recursive=1"
        try:
            response = requests.get(api_url, timeout=10)
            if response.status_code == 200:
                tree = response.json().get('tree', [])
                for item in tree:
                    if re.search(pattern, item['path'], re.IGNORECASE):
                        return True
        except requests.RequestException:
            pass
        return False

    def is_git_repo(self, uri):
        return any(host in uri for host in ["github.com", "gitlab.com", "bitbucket.org"])

    def is_python_repo(self, uri):
        return (
            self.check_file_exists(uri, "setup.py") or
            self.check_file_exists(uri, "pyproject.toml") or
            self.check_file_exists(uri, "Pipfile") or
            self.check_file_exists(uri, "environment.yml") or
            self.check_contains_filetype(uri, r"\.py$")
        )

    def check_reproducibility(self):
        for artifact_id, uri in self.repositories.items():
            python_based = self.is_python_repo(uri)
            results = {
                "Python-based": python_based,
                "README.md": self.check_file_exists(uri, "README.md"),
                "LICENSE": self.check_file_exists(uri, "LICENSE") or self.check_file_exists(uri, "LICENSE.txt"),
                "environment.yml": self.check_file_exists(uri, "environment.yml"),
                "Dockerfile": self.check_file_exists(uri, "Dockerfile"),
                "setup.py or pyproject.toml": (
                    self.check_file_exists(uri, "setup.py") or self.check_file_exists(uri, "pyproject.toml")
                ),
                "CI config (.github/workflows/, .gitlab-ci.yml)": (
                    self.check_contains_filetype(uri, r"\.github/workflows/.*\.yml$") or
                    self.check_file_exists(uri, ".gitlab-ci.yml")
                ),
                "Jupyter Notebooks (*.ipynb)": self.check_contains_filetype(uri, r".*\.ipynb$"),
                "CITATION.cff": self.check_file_exists(uri, "CITATION.cff"),
                "Version Control (.git)": self.is_git_repo(uri),
            }

            print(f"\n📦 Artifact {artifact_id} ({uri}):")
            for key, val in results.items():
                print(f" - {key}: {'✅' if val else '❌'}")

if __name__ == "__main__":
    json_file_path = 'artifacts.json'
    checker = ReproducibilityChecker(json_file_path)
    checker.check_reproducibility()



📦 Artifact artifact_1 (https://github.com/hihey54/hicss58):
 - Python-based: ✅
 - README.md: ✅
 - LICENSE: ✅
 - environment.yml: ❌
 - Dockerfile: ❌
 - setup.py or pyproject.toml: ❌
 - CI config (.github/workflows/, .gitlab-ci.yml): ❌
 - Jupyter Notebooks (*.ipynb): ❌
 - CITATION.cff: ❌
 - Version Control (.git): ✅

📦 Artifact artifact_2 (https://github.com/githubkilroy/TCN-Dataset):
 - Python-based: ❌
 - README.md: ❌
 - LICENSE: ❌
 - environment.yml: ❌
 - Dockerfile: ❌
 - setup.py or pyproject.toml: ❌
 - CI config (.github/workflows/, .gitlab-ci.yml): ❌
 - Jupyter Notebooks (*.ipynb): ❌
 - CITATION.cff: ❌
 - Version Control (.git): ✅

📦 Artifact artifact_3 (https://github.com/TheAlgorithms/Python):
 - Python-based: ❌
 - README.md: ❌
 - LICENSE: ❌
 - environment.yml: ❌
 - Dockerfile: ❌
 - setup.py or pyproject.toml: ❌
 - CI config (.github/workflows/, .gitlab-ci.yml): ❌
 - Jupyter Notebooks (*.ipynb): ❌
 - CITATION.cff: ❌
 - Version Control (.git): ✅
